In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
from itertools import combinations

# ----------------- CONFIG -----------------
# Path to input CSV folder (update if needed)
INPUT_DIR = "/content/drive/MyDrive/Expression_Raw_Data_IDH1status/"
OUTPUT_DIR = "/content/drive/MyDrive/Expression_Raw_Data_IDH1status_REPEAT/"

# Define a mapping for various input group labels to canonical output labels
GROUP_LABEL_MAP = {
    "idhwt": "IDHwt",
    "wildtype": "Wildtype",
    "mutant": "Mutant",
    "idhmut-codel": "IDHmut-codel",
    "idhmut-non-codel": "IDHmut-non-codel",
    "idhwt-codel": "IDHwt-codel",
    "idhwt-non-codel": "IDHwt-non-codel"
}

# Expected group labels for column detection (derived from keys of the map, for case-insensitive matching)
GROUP_LABELS = set(GROUP_LABEL_MAP.keys())

# Define the canonical order for plotting. This should include all possible output group names
# and specify their desired order. Simpler categories like 'Wildtype' and 'Mutant' are placed first.
GLOBAL_PLOT_ORDER = ["Wildtype", "Mutant", "IDHmut-codel", "IDHmut-non-codel", "IDHwt-codel", "IDHwt-non-codel", "IDHwt"]

# Define a color palette that matches the GLOBAL_PLOT_ORDER.
# If a group is not present in the current file, its color will simply not be used.
# The first two colors are assigned to 'Wildtype' and 'Mutant' for general cases,
# followed by the specific IDH subtype colors.
GLOBAL_PALETTE = {
    "Wildtype": "#0072B2",  # Blue for Wildtype
    "Mutant": "#D55E00",    # Orange for Mutant
    "IDHmut-codel": "#d95f5f",
    "IDHmut-non-codel": "#2ca02c",
    "IDHwt-codel": "#4c72b0",
    "IDHwt-non-codel": "#ff7f0e",
    "IDHwt": "#4C72B0"
}

# Plot styling
sns.set(style="ticks")
FIGSIZE = (6, 8)
dpi = 600

# Create output dir if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------- Helpers -----------------
def find_group_column(df):
    """Find column in df that contains group labels matching GROUP_LABELS (case-insensitive)."""
    for col in df.columns:
        vals = df[col].astype(str).str.strip()
        # normalize values
        normalized = set(v.lower() for v in vals.unique())
        target_norm = set(v.lower() for v in GROUP_LABELS)
        # check if intersection is large enough
        if len(normalized & target_norm) >= 2:
            return col
    # fallback: look for columns with substring 'idh' or 'subtype'
    for col in df.columns:
        if 'idh' in col.lower() or 'subtype' in col.lower():
            return col
    return None

def find_expression_column(df, group_col):
    """Pick a numeric column (not the group column). If many, pick the one with largest variance."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if group_col in numeric_cols:
        numeric_cols = [c for c in numeric_cols if c != group_col]
    if len(numeric_cols) == 0:
        # sometimes numeric stored as strings, attempt to coerce
        candidates = [c for c in df.columns if c != group_col]
        variances = {}
        for c in candidates:
            coerced = pd.to_numeric(df[c], errors='coerce')
            if coerced.notna().sum() > 5:
                variances[c] = coerced.var()
        if variances:
            return max(variances, key=variances.get)
        return None
    if len(numeric_cols) == 1:
        return numeric_cols[0]
    # choose numeric col with highest variance
    variances = {c: df[c].var() for c in numeric_cols}
    return max(variances, key=variances.get)

def p_to_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return 'ns'

def draw_bracket(ax, x1, x2, y, h, text):
    """Draw bracket line and text between x1 and x2 at height y. x positions are category indices (0-based)."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.0, c='black')
    ax.text((x1+x2)/2.0, y+h, text, ha='center', va='bottom', fontsize=11)

# ----------------- Main loop over files -----------------
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))
if len(csv_files) == 0:
    print(f"No CSV files found in {INPUT_DIR}. Please check path or upload CSVs.")
else:
    print(f"Found {len(csv_files)} CSV files. Processing...")

for fpath in csv_files:
    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        print(f"Skipping {fpath}: cannot read CSV ({e})")
        continue

    # gene name from filename (without extension)
    gene = os.path.splitext(os.path.basename(fpath))[0]

    # detect group and expression columns
    group_col = find_group_column(df)
    expr_col = find_expression_column(df, group_col)

    if group_col is None or expr_col is None:
        print(f"[{gene}] Could not detect group or expression column automatically. Skipping file.")
        continue

    # prepare tidy dataframe
    plot_df = df[[group_col, expr_col]].copy()
    plot_df.columns = ["group", "expression"]
    # drop missing
    plot_df = plot_df.dropna(subset=["group", "expression"])
    # coerce numeric expression if necessary
    plot_df['expression'] = pd.to_numeric(plot_df['expression'], errors='coerce')
    plot_df = plot_df.dropna(subset=['expression'])

    # Filter groups to just the recognized ones and preserve order
    # Use the canonical map to normalize group names
    def canonicalize(g):
        return GROUP_LABEL_MAP.get(str(g).strip().lower(), str(g).strip())

    plot_df['group'] = plot_df['group'].apply(canonicalize)

    # Determine the present groups and their order for the current plot
    present_groups = [g for g in GLOBAL_PLOT_ORDER if g in plot_df['group'].unique()]
    if len(present_groups) < 2:
        print(f"[{gene}] Not enough recognized groups (found {plot_df['group'].unique()}). Skipping.")
        continue

    # Create a palette for the present groups only
    current_palette = [GLOBAL_PALETTE[g] for g in present_groups if g in GLOBAL_PALETTE]

    # set up the plot
    plt.figure(figsize=FIGSIZE, dpi=dpi)
    ax = sns.boxplot(x='group', y='expression', data=plot_df, order=present_groups,
                     palette=current_palette, fliersize=0, width=0.6)
    # jittered points (swarmplot)
    sns.stripplot(x='group', y='expression', data=plot_df, order=present_groups,
                  color='black', size=4, jitter=0.25, alpha=0.8)

    # Titles and labels
    ax.set_title(f"{gene}", fontsize=16)
    ax.set_xlabel("")
    ax.set_ylabel("mRNA expression (log2)", fontsize=12, fontweight='bold')

    # remove rotation of x labels
    plt.setp(ax.get_xticklabels(), rotation=0, ha='center', fontweight='bold')

    # set tick line width for x and y major ticks
    ax.tick_params(axis='both', width=2, color='black')

    # put sample-size text inside each box (centered at the group median)
    # place sample-size text on the bottom boundary of the axes (aligned under each category)
    group_counts = plot_df.groupby('group').size().reindex(present_groups)

    # use the x-axis transform so x is data coords but y is axis fraction (0..1)
    # y = -0.06 places the text a little below the x-axis inside the figure margin;
    # make this more negative (e.g. -0.09) to push further down, or closer to 0 to move up.
    # y_axis_frac = 0.06
    # for i, g in enumerate(present_groups):
    #     ax.text(i, y_axis_frac, f"n = {group_counts[g]}",
    #             ha='center', va='top',
    #             fontsize=11, fontstyle='italic', fontweight='bold',
    #             transform=ax.get_xaxis_transform(),  # x in data coords, y in axis coords
    #             clip_on=False)  # allow drawing outside the axes box if needed

    # Pairwise significance tests and bracket placement
    # Requires statsmodels; in Colab run once: !pip install statsmodels
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from scipy import stats

    # Build pairs in canonical order for all combinations
    pairs = list(combinations(present_groups, 2))
    # Ensure consistent ordering for display, e.g., by the order in 'present_groups'
    pairs = sorted(pairs, key=lambda p: (present_groups.index(p[0]), present_groups.index(p[1])))

    # Run one-way ANOVA across the present groups (only if >=2 groups)
    anova_p = None
    if len(present_groups) >= 2:
        group_data = [plot_df.loc[plot_df['group'] == g, 'expression'].values for g in present_groups]
        try:
            anova_res = stats.f_oneway(*group_data)
            anova_p = anova_res.pvalue
        except Exception:
            anova_p = None

    # Run Tukey HSD on the raw data (this provides pairwise adjusted p-values)
    tukey_pmap = {}  # map tuple(sorted(g1,g2)) -> p_adj
    if len(plot_df['group'].unique()) >= 2:
        try:
            tukey = pairwise_tukeyhsd(endog=plot_df['expression'], groups=plot_df['group'], alpha=0.05)
            # tukey._results_table.data is a list of rows; header = row0, data rows follow
            rows = tukey._results_table.data[1:]  # skip header
            for r in rows:
                g1 = r[0]
                g2 = r[1]
                p_adj = float(r[3])  # column 'p-adj'
                key = (g1, g2)
                tukey_pmap[key] = p_adj
                # also add reversed key for convenience
                tukey_pmap[(g2, g1)] = p_adj
        except Exception:
            tukey_pmap = {}

    # Convert group-name pairs to x-index pairs and p-values list in same order
    x_pairs = []
    pvals = []
    for (g1, g2) in pairs:
        x1 = present_groups.index(g1)
        x2 = present_groups.index(g2)
        x_pairs.append((x1, x2))
        # get adjusted p from tukey map if available, else fall back to t-test pvalue
        p_adj = tukey_pmap.get((g1, g2), None)
        if p_adj is None:
            # fallback to Welch t-test
            data1 = plot_df.loc[plot_df['group'] == g1, 'expression'].values
            data2 = plot_df.loc[plot_df['group'] == g2, 'expression'].values
            try:
                p_adj = ttest_ind(data1, data2, equal_var=False, nan_policy='omit').pvalue
            except Exception:
                p_adj = 1.0
        pvals.append(p_adj)

    # Now draw brackets using axis-range based heights (robust and consistent)
    ylim = ax.get_ylim()
    y_range = ylim[1] - ylim[0]
    top = ylim[1]

    # Sort by span (longer first) to make stacking nicer
    order = sorted(range(len(x_pairs)), key=lambda i: (x_pairs[i][1] - x_pairs[i][0]), reverse=True)

    placed = []
    step = 0.1    # vertical step (fraction of y_range) between stacked bars
    tip_frac = 0.02

    for idx in order:
        x1, x2 = x_pairs[idx]
        p = pvals[idx]

        # stack level = how many placed bars overlap this x-range
        level = sum(1 for (bx1, bx2, by) in placed if not (x2 < bx1 or x1 > bx2))

        bar_height = top + (y_range * step * (level + 1))
        bar_tips = bar_height - (y_range * tip_frac)

        # Draw bracket on correct axes; clip_on=False so it's visible even slightly outside axes
        ax.plot([x1, x1, x2, x2],
                [bar_tips, bar_height, bar_height, bar_tips],
                lw=1.2, c='k', clip_on=False)

        # translate p to stars
        if p < 0.001:
            sig_symbol = '***'
        elif p < 0.01:
            sig_symbol = '**'
        elif p < 0.05:
            sig_symbol = '*'
        else:
            sig_symbol = 'ns'

        text_height = bar_height + (y_range * 0.01)
        ax.text((x1 + x2) * 0.5, text_height, sig_symbol,
                ha='center', va='bottom', c='k', fontsize=12, clip_on=False)

        placed.append((x1, x2, bar_height))

    # Expand y-limits so bars and stars are visible
    if placed:
        highest = max(by for (_, _, by) in placed)
        extra = y_range * 0.05
        ax.set_ylim(ylim[0], max(ax.get_ylim()[1], highest + extra))

    # Optionally, you can annotate the ANOVA p-value somewhere (e.g., top-left)
    if anova_p is not None:
        # small note at top-left inside axes, adjust fractional position with transform
        ax.text(0.01, 0.98, f"ANOVA p = {anova_p:.3g}", transform=ax.transAxes,
                ha='left', va='top', fontsize=9)

    plt.tight_layout()

    # save
    outname = os.path.join(OUTPUT_DIR, f"{gene}_boxplot.png")
    plt.savefig(outname, dpi=dpi)
    plt.close()
    print(f"Saved: {outname}")

print("All done.")

Found 3 CSV files. Processing...


/tmp/ipython-input-3827117020.py:161: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x='group', y='expression', data=plot_df, order=present_groups,


Saved: /content/drive/MyDrive/Expression_Raw_Data_IDH1status_REPEAT/CHPT1_boxplot.png


/tmp/ipython-input-3827117020.py:161: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x='group', y='expression', data=plot_df, order=present_groups,


Saved: /content/drive/MyDrive/Expression_Raw_Data_IDH1status_REPEAT/CSAD_boxplot.png


/tmp/ipython-input-3827117020.py:161: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(x='group', y='expression', data=plot_df, order=present_groups,


Saved: /content/drive/MyDrive/Expression_Raw_Data_IDH1status_REPEAT/CTH_boxplot.png
All done.
